In [1]:
##### Compiles rasters of all predictors into one file

import os
import pandas as pd
import rasterio
from pathlib import Path
import numpy as np


In [2]:
### Set-up

# Get the current working directory
cd = Path.cwd().parent 

# folder of predictors 
predictors = Path(f'{cd}/Data/Clean/Predictors/Rasters')
tif_files  = sorted(predictors.glob("*.tif"))

# USD production 
production_path = f'{cd}/Data/Clean/Production/total_production_USD_2020.tif'

# save path
save_path = f'{cd}/Data/Clean/Predictors/raster_matrix_production_only.parquet'

In [3]:
#### Combine all rasters

# read rasters
arrays = {}
coords = None

for path in tif_files:
    with rasterio.open(path) as src:
        arr    = src.read(1).astype(np.float32)
        nodata = src.nodata
        if nodata is not None:
            arr[arr == nodata] = np.nan

        if coords is None:
            height, width = arr.shape
            transform = src.transform
            rows, cols = np.meshgrid(np.arange(height), np.arange(width), indexing="ij")
            xs, ys = rasterio.transform.xy(transform, rows.ravel(), cols.ravel())
            coords = pd.DataFrame({"x": xs, "y": ys})

        arrays[path.stem] = arr.ravel()

# add production raster
with rasterio.open(production_path) as src:
    arr    = src.read(1).astype(np.float32)
    nodata = src.nodata
    if nodata is not None:
        arr[arr == nodata] = np.nan
    arrays["total_production_USD_2020"] = arr.ravel()

# compile into dataframe
all_rasters = pd.concat([coords, pd.DataFrame(arrays)], axis=1)

# drop if terrain is missing (selects only land pixels)
all_rasters = all_rasters.dropna(subset=["terrain_slope"])
all_rasters = all_rasters.dropna(subset=["total_production_USD_2020"])
all_rasters = all_rasters[all_rasters['total_production_USD_2020'] != 0]

# save
all_rasters.to_parquet(save_path, index=False)